# M4 · 快算 —— 注意力后端与 INT8 权重

**学习目标：** 在你的 W7900 上按规定的 40 步负载验证两个*快算*手段：F1 = ck 注意力内核，F2 = INT8 量化 DiT 权重。两个臂只改变执行速度、不改变采样量——因此质量门采用**保真语义**。测得退化是有效的科学结果，必须保留展示。

**先预测：** 参考主机上 F1 ≈ 45.1 秒、F2 ≈ 51.1 秒中位数——但注意力后端的效果高度依赖架构；请写下你自己的预期。

参考讲义：[M04 讲义](../course/M04_faster_computation/LESSON.zh-CN.md) · [English](./04_faster_computation.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="04")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

## 1 · 在**你的** W7900 上运行 F1（ck 注意力）与 F2（INT8）
B0 来自 M1/本轮实验（显示 run_id）。每个臂都会以锁定的服务端参数启动全新服务器——如果注意力后端或模型发生静默回退，运行器会拒绝测量。

In [ ]:
def run_faster_arms():
    prof = "classroom" if session.mode == "GPU_CLASSROOM" else "research"
    b0 = session.run("B0", profile=prof)
    f1 = session.run("F1", profile=prof)
    f2 = session.run("F2", profile=prof)
    return b0, f1, f2

arms = gpu_cell(run_faster_arms)
if arms is None:
    ids = [session.latest_campaign_run(x) for x in ("B0", "F1", "F2")]
    if all(ids):
        arms = tuple(session.consume(i, why=f"campaign {x}")
                     for i, x in zip(ids, ("B0", "F1", "F2")))
if arms is not None:
    ab_f1 = rt.ab_compare(arms[0], arms[1], label="B0 vs F1 (ck attention)")
    ab_f2 = rt.ab_compare(arms[0], arms[2], label="B0 vs F2 (int8 weights)")
    for ab in (ab_f1, ab_f2):
        print({k: ab[k] for k in ("classification", "speedup_median",
                                  "latency_delta_s", "vram_delta_mb")})

## 2 · 验证优化确实生效
如果服务器静默使用了默认后端，所谓的注意力收益毫无意义。记录中带有从服务器日志抓取的证据——解读加速比之前先断言它。

In [ ]:
def attention_evidence(bundle):
    evs = [r.get("attention_evidence") or {} for r in bundle.records]
    active = [e for e in evs if e]
    if not active:
        return "NO_EVIDENCE_IN_RECORDS"
    return active[-1]

if arms is not None:
    for name, b in (("F1", arms[1]), ("F2", arms[2])):
        st = b.stats()
        print(f"{name}: attn={st['attention_backend']} unet={st['unet']} "
              f"precision={st['precision']}")
        print(f"   server-log evidence: {attention_evidence(b)}")
    b0s = arms[0].stats()
    print("B0 attention:", b0s["attention_backend"],
          "| F1 must show the ck backend ACTIVE, F2 the int8 unet file")

## 3 · 诚实解读数字
1. 永远不要把独立加速比相乘：实测 F1×F2 ≠ 实测组合效果（交互是真实的；那是 M6 的工作）。
2. INT8 的收益常在 **VRAM** 而非原始速度——两者都要量化。
3. 如果你测得 F1 是退化，就按实测报告。

In [ ]:
if arms is not None:
    b0s, f1s, f2s = (a.stats() for a in arms)
    print(f"{'arm':4s} {'median_s':>9s} {'n':>3s} {'vram_dev_mb':>11s}")
    for nm, st in (("B0", b0s), ("F1", f1s), ("F2", f2s)):
        print(f"{nm:4s} {st['median_s']:9.2f} {st['n_warm']:3d} "
              f"{st['vram_device_peak_mb'] or st['vram_peak_mb']:11.0f}")
    print("\nINT8 VRAM saving (MB):",
          (b0s['vram_device_peak_mb'] or b0s['vram_peak_mb'])
          - (f2s['vram_device_peak_mb'] or f2s['vram_peak_mb']))
    for ab, nm in ((ab_f1, "F1"), (ab_f2, "F2")):
        _stamp(MEASURED_TODAY,
               f"{nm} speedup {ab['speedup_median']}x [{ab['classification']}]")
    print("reminder: do NOT report F1*F2 as a combined measured gain")

## 4 · 质量门的保真语义
F1/F2 与 B0 保持相同的 40 步调度，因此成对图像按*数值变体*检查（LPIPS 保真阈值用于分流人工评审；它绝不会自动通过）。

> **质量判定永远由人完成。** 任何单元都不会自动批准图像质量结论。历史所有者判定**只**适用于其原始评估的部署范围；新的物理运行需要自己的评审（在此之前保持 `PENDING_HUMAN_REVIEW`）。

In [ ]:
print("run your own fidelity gate in GPU modes:")
print("  session.run_quality('F1', reference='B0')")
print("  session.run_quality('F2', reference='B0')")
print("frozen campaign artifacts: evidence/quality/F1, evidence/quality/F2")
_stamp(HISTORICAL, "reference F2 fidelity gate triaged human review; INT8 "
      "numerical FAIL (in M5.1's H3_OV_INT8 encoder) stays FAIL - never promoted")

## 5 · 动手练习
1. 解释为什么 ck 注意力可能只在某些 ROCm 架构上有收益。
2. 参考主机上 INT8 降低了约 6.6 GB VRAM——这在 48 GB 显卡上能带来什么？
3. 设计一个能分离 F2 中*反量化开销*的单臂实验。

## 验收标准

- [ ] 你能指出哪些数字是**今天在这块 GPU 上实测的**、哪些是
      **历史参考值**、哪些是**估计/模拟**。
- [ ] 你产生的每次物理运行都在 `runs/` 下有可见的 `run_id`。
- [ ] 你没有把 smoke/课堂样本冒充为正式协议结果。
- [ ] 你能解释你报告的每项测量的一个局限性。

## 复现与贡献

```bash
python -m lab doctor                                  # 准入证据 -> runs/doctor.json
python -m lab run --experiment F1 --date <your-tag>  # 完整研究协议
python -m lab report --scope student --kind all       # 只汇总你自己的 runs
```

提交你的 `runs/<run-id>/*.json` 记录（绝不写 `evidence/`），并在 PR 中说明：
假设 → 协议 → 数字 → 局限。欢迎负结果——测得的退化是科学结果，不是失败。

```bash
python -m lab run --experiment F2 --date <your-tag>   # INT8 arm
```